# Data Preparation 

In [92]:
# import libs
import pandas as pd
import numpy as np

In [93]:
# import dataset

df_orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
df_order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
df_customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
df_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
df_products = pd.read_csv('../data/raw/olist_products_dataset.csv')
df_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')
df_geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
df_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
df_sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

## 1. orders table

### 1.1 Create Working Copy

In [94]:
orders_clean = df_orders.copy()

### 1.2 Convert Data Types and Validate Conversion

In [95]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
              'order_delivered_carrier_date', 'order_delivered_customer_date',
              'order_estimated_delivery_date']
orders_clean[date_cols] = orders_clean[date_cols].apply(pd.to_datetime)

In [96]:
orders_clean.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

### 1.3 Missing Values After Datetime Conversion

In [97]:
orders_clean.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [98]:
# Duplicate Validation
orders_clean.order_id.duplicated().sum()

np.int64(0)

### 1.4 Business Rules Validation

In [99]:
orders_clean[
    (orders_clean['order_delivered_carrier_date'].isnull()) & 
    (orders_clean['order_delivered_customer_date'].notnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14


In [100]:
orders_clean[
    (orders_clean['order_approved_at'].isnull()) & 
    (orders_clean['order_delivered_carrier_date'].notnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


In [101]:
orders_clean[
    (orders_clean['order_status'] == 'delivered') & 
    (orders_clean['order_delivered_customer_date'].isnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


### 1.5 Feature Engineering

In [102]:
orders_clean['purchase_to_approved_minutes'] = (
    (orders_clean['order_approved_at'] - orders_clean['order_purchase_timestamp'])
    .dt.total_seconds() / 60
)


orders_clean['approval_to_carrier_minutes'] = (
    (orders_clean['order_delivered_carrier_date'] - orders_clean['order_approved_at'])
    .dt.total_seconds() / 60
)

orders_clean['carrier_to_customer_minutes'] = (
    (orders_clean['order_delivered_customer_date'] - orders_clean['order_delivered_carrier_date'])
    .dt.total_seconds() / 60
)


orders_clean['purchase_to_customer_minutes'] = (
    (orders_clean['order_delivered_customer_date'] - orders_clean['order_purchase_timestamp'])
    .dt.total_seconds() / 60
)


orders_clean['purchase_to_estimated_minutes'] = (
    (orders_clean['order_estimated_delivery_date'] - orders_clean['order_purchase_timestamp'])
    .dt.total_seconds() / 60
)

In [103]:
orders_clean[['purchase_to_approved_minutes','approval_to_carrier_minutes',
              'carrier_to_customer_minutes','purchase_to_customer_minutes',
              'purchase_to_estimated_minutes']].agg(['min', 'max'])

,purchase_to_approved_minutes,approval_to_carrier_minutes,carrier_to_customer_minutes,purchase_to_customer_minutes,purchase_to_estimated_minutes
min,0.000000,-246555.366667,-23178.483333,768.116667,2374.550000
max,270550.833333,181098.100000,295475.000000,301865.200000,223395.066667


### 1.6 Final Quality Check

#### Data Quality Summary

- Date columns successfully converted to datetime.
- No duplicate order IDs found.
- Missing delivery dates are consistent with non-delivered orders.
- Business rule validation did identify critical inconsistencies.
- Duration features successfully created.

#### Decisions

- Keep missing delivery timestamps because they represent valid business scenarios.
- No duplicate records removed.
- No missing values were imputed.

### 1.7 Export Prepared Dataset

In [104]:
orders_clean.to_csv('../data/processed/olist_orders_cleaned.csv', index=False)

## 2. order items

### 2.1 Create Working Copy

In [105]:
order_items_clean = df_order_items.copy()

### 2.2 Convert Data Types and Validate Conversion


In [106]:
order_items_date_columns = ['shipping_limit_date']
order_items_clean[order_items_date_columns] = order_items_clean[order_items_date_columns].apply(pd.to_datetime)

In [107]:
order_items_clean.dtypes

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

### 2.3 Missing Values After Datetime Conversion

In [108]:
order_items_clean.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [109]:
# Duplicate Validation
order_items_clean.duplicated().sum()

np.int64(0)

In [110]:

order_items_clean.duplicated(subset=['order_id', 'order_item_id']).sum()

np.int64(0)

### 2.4 Business Rules Validation

In [111]:
order_items_clean[['order_item_id','price','freight_value']].agg(['min', 'max'])

,order_item_id,price,freight_value
min,1,0.85,0.00
max,21,6735.00,409.68


In [112]:
min_per_order = order_items_clean.groupby('order_id')['order_item_id'].min()

(min_per_order != 1).sum()

np.int64(0)

In [113]:
def is_consecutive(series):
    vals = sorted(series.tolist()) 
    expected = list(range(1, len(vals) + 1)) 
    return vals == expected

check = order_items_clean.groupby('order_id')['order_item_id'].apply(is_consecutive)
(~check).sum()  

np.int64(0)

### 2.6 Final Quality Check
- converted shipping date to datetime
- no missing values were removed
- no duplicate records were removed
- composite key uniqueness was preserved
- all order_items start as 1 
- All order_items were continuous

### 2.7 Export Prepared Dataset

In [114]:
# order_items_clean.to_csv('../data/processed/olist_order_items_cleaned.csv', index=False)

## 3. customers table

### 3.1 Create Working Copy

In [115]:
customers_clean = df_customers.copy()

### 3.2 Convert Data Types and Validate Conversion
Checked.
No conversions were required.


### 3.3 Duplicate Validation

In [142]:
customers_clean.duplicated().sum()

np.int64(0)

In [143]:
customers_clean.customer_id.duplicated().sum()

np.int64(0)

### 3.4 Business Rules Validation

In [ ]:
customers_clean.groupby('customer_unique_id')['customer_zip_code_prefix'].nunique().value_counts()

customer_zip_code_prefix
1    95846
2      246
3        4
Name: count, dtype: int64

In [ ]:
customers_clean.groupby('customer_unique_id')['customer_city'].nunique().value_counts()

customer_city
1    95974
2      121
3        1
Name: count, dtype: int64

### 3.6 Final Quality Check

#### Data Quality Summary

- no missing values detected
- no duplicate customer IDs found
- Data types required no modification
- customer unique IDs correctly represent repeat customers
- customer IDs are unique

#### Cleaning Decisions

- no records were removed
- no missing values required imputation
- no data type conversion was necessary
- repeated customer_unique_id values were retained because they represent multiple purchases made by the same customer

### 3.7 Export Prepared Dataset

In [ ]:
# customers_clean.to_csv('../data/processed/olist_customers_cleaned.csv', index=False)